In [1]:
from Stream1_DataExtraction import *

dataset = load_dataset("wine", seed = 42)

print(dataset.categorical_columns)

[]


In [2]:
from Stream2_Preprocessing import *

prep = TabularPreprocessor(dataset.numerical_columns, dataset.categorical_columns)
prep.fit(dataset.X_train)

In [3]:
train = prep.transform(dataset.X_train, dataset.y_train)

In [4]:
test = prep.transform(dataset.X_test, dataset.y_test)

In [5]:
val = prep.transform(dataset.X_val, dataset.y_val)

In [6]:
torch.manual_seed(42)
n_num = train.x_num.shape[1] if train.x_num is not None else 0
n_cat = train.x_cat.shape[1] if train.x_cat is not None else 0
embedder = PLREmbeddings(n_num) if n_num else None

In [7]:
from Stream3_ModernNCA import *

model = ModernNCAEncoder(
    plr_module=embedder,
    num_continuous=n_num,
    plr_dim=embedder.linear.out_features if embedder is not None else 1,
    num_categorical=n_cat,
    hidden_dim=32,
    embed_dim=64,
    num_layers=1,
    dropout_rate=0.1,
    post_mlp_layers=[],
)

In [8]:
from Stream4_Train import *

trainer = Trainer(model, config={})
trainer.fit((train.x_num, train.x_cat, train.y), 4, 0.3, dataset.task_type, dataset.n_classes, (val.x_num, val.x_cat, val.y))

None
Epoch 1 val Loss: 0.687139093875885
Epoch 2 val Loss: 0.681205689907074


Epoch 3 val Loss: 0.6733871698379517
Epoch 4 val Loss: 0.6631176471710205


In [9]:
print(train.x_cat)

None


In [10]:
from steram5 import compute_metric

predictions = trainer.predict(
    (train.x_num, train.x_cat, train.y),
    (test.x_num, test.x_cat),
    dataset.task_type,
    dataset.n_classes,
)

if dataset.task_type in ("binary", "multiclass"):
    predictions = predictions.argmax(dim=1)

metric_name, metric_value = compute_metric(
    dataset.task_type,
    test.y.cpu().numpy(),
    predictions.cpu().numpy(),
)
print(metric_name, metric_value)

accuracy 0.701171875


In [11]:
from steram5 import *

results = run_experiments(
    model_names=["catboost", "xgboost"],
    datasets=[dataset],
    seeds=[42],
    preprocessor=prep,
    n_trials=10,
)

print(results[["dataset", "model", "seed", "metric", "metric_value"]])

[I 2026-10-05 15:39:14,500] A new study created in memory with name: no-name-9a732e8c-a6e2-4a97-9c93-d4f499d0b7ac


[I 2026-10-05 15:39:21,432] Trial 0 finished with value: 0.7377450980392157 and parameters: {'bagging_temperature': 0.5488135039273248, 'depth': 8, 'l2_leaf_reg': 4.006483661964395, 'leaf_estimation_iterations': 6, 'learning_rate': 0.0013130280280658588}. Best is trial 0 with value: 0.7377450980392157.


[I 2026-10-05 15:39:25,034] Trial 1 finished with value: 0.7279411764705882 and parameters: {'bagging_temperature': 0.6458941130666561, 'depth': 6, 'l2_leaf_reg': 7.79422611516095, 'leaf_estimation_iterations': 10, 'learning_rate': 0.0008264328927007723}. Best is trial 0 with value: 0.7377450980392157.


[I 2026-10-05 15:39:29,632] Trial 2 finished with value: 0.7303921568627451 and parameters: {'bagging_temperature': 0.7917250380826646, 'depth': 7, 'l2_leaf_reg': 3.6986612821028797, 'leaf_estimation_iterations': 10, 'learning_rate': 2.2655846366566637e-05}. Best is trial 0 with value: 0.7377450980392157.


[I 2026-10-05 15:39:31,757] Trial 3 finished with value: 0.7279411764705882 and parameters: {'bagging_temperature': 0.08712929970154071, 'depth': 3, 'l2_leaf_reg': 6.801737161546002, 'leaf_estimation_iterations': 8, 'learning_rate': 0.22390342721683687}. Best is trial 0 with value: 0.7377450980392157.


...

[I 2026-10-05 15:40:36,137] Trial 6 finished with value: 0.7205882352941176 and parameters: {'use_reg_alpha': True, 'reg_alpha': 4.3800086026247575e-08, 'colsample_bylevel': 0.8462360596850099, 'colsample_bytree': 0.7833007271032876, 'use_gamma': True, 'gamma': 8.697713691175405e-08, 'use_reg_lambda': True, 'reg_lambda': 1.533520282967531e-05, 'learning_rate': 0.021729608403132785, 'max_depth': 4, 'min_child_weight': 20.52360528194992, 'subsample': 0.6447030464736005}. Best is trial 3 with value: 0.7794117647058824.


[I 2026-10-05 15:40:40,354] Trial 7 finished with value: 0.7720588235294118 and parameters: {'use_reg_alpha': True, 'reg_alpha': 1.5888227943138278e-08, 'colsample_bylevel': 0.9144700146086815, 'colsample_bytree': 0.5023477380962735, 'use_gamma': False, 'use_reg_lambda': True, 'reg_lambda': 3.0727797718456485e-06, 'learning_rate': 0.0075995289207579165, 'max_depth': 7, 'min_child_weight': 0.2749633277888864, 'subsample': 0.6115408163203091}. Best is trial 3 with value: 0.7794117647058824.


[I 2026-10-05 15:40:43,882] Trial 8 finished with value: 0.7401960784313726 and parameters: {'use_reg_alpha': False, 'colsample_bylevel': 0.923204336235564, 'colsample_bytree': 0.8497396376587522, 'use_gamma': True, 'gamma': 9.226933879829086e-05, 'use_reg_lambda': False, 'learning_rate': 0.25625763237199567, 'max_depth': 8, 'min_child_weight': 26.810544711184114, 'subsample': 0.7506621909633511}. Best is trial 3 with value: 0.7794117647058824.


[I 2026-10-05 15:40:47,688] Trial 9 finished with value: 0.7450980392156863 and parameters: {'use_reg_alpha': False, 'colsample_bylevel': 0.7119275242790899, 'colsample_bytree': 0.8031966070639622, 'use_gamma': True, 'gamma': 0.039970112895606855, 'use_reg_lambda': True, 'reg_lambda': 0.00019394876095968973, 'learning_rate': 4.757407151830433e-05, 'max_depth': 5, 'min_child_weight': 0.2567697386862121, 'subsample': 0.7954363806240866}. Best is trial 3 with value: 0.7794117647058824.


  dataset     model  seed    metric  metric_value
0    wine  catboost    42  accuracy      0.751953
1    wine   xgboost    42  accuracy      0.775391
